# 🦜🔗 LangChain + Mistral AI — Agente Pandas con tu propio Dataset

Este notebook te permite:
- 📦 Instalar LangChain + Mistral AI
- 🔑 Configurar tu API Key de Mistral
- 📂 Cargar tu propio dataset (CSV, Excel, JSON)
- 🤖 Crear un **Agente Pandas** con `create_pandas_dataframe_agent`
- 🧠 Entender cómo funciona `.invoke()` en LangChain
- 💬 Hacerle preguntas en lenguaje natural a tus datos

---
> **Modelo usado:** `mistral-large-latest` vía LangChain  
> **Agente:** `create_pandas_dataframe_agent` (LangChain experimental)

## 📦 PASO 1 — Instalación de dependencias

In [1]:
%%capture
!pip install langchain langchain-mistralai langchain-experimental
!pip install pandas openpyxl tabulate --upgrade

print("✅ Instalación completada")

## 🔑 PASO 2 — Configurar tu API Key de Mistral AI

Obtén tu clave en: https://console.mistral.ai/

In [10]:
import os
from getpass import getpass

# Ingresa tu API Key de forma segura
MIAPIKEI = getpass("🔑 Ingresa tu Mistral API Key: ")
os.environ["MIAPIKEI"] = MIAPIKEI

print("✅ API Key configurada correctamente")

🔑 Ingresa tu Mistral API Key: ··········
✅ API Key configurada correctamente


## 🧠 PASO 3 — Inicializar el modelo Mistral con LangChain

### ¿Qué es `.invoke()`?

En LangChain, **`.invoke()`** es el método estándar para ejecutar cualquier componente:

```
componente.invoke(input)  →  output
```

| Componente | `.invoke()` recibe | `.invoke()` devuelve |
|---|---|---|
| `LLM` | `str` | `AIMessage` |
| `Chain` | `dict` | `dict` |
| `Agent` | `dict` con `"input"` | `dict` con `"output"` |
| `Prompt` | `dict` | `PromptValue` |

Es parte de la interfaz **Runnable** — todos los objetos LangChain la implementan.

In [11]:
from langchain_mistralai import ChatMistralAI

# ── Inicializar el modelo ──────────────────────────────────────────────
llm = ChatMistralAI(
    model="mistral-large-latest",   # Modelo más capaz de Mistral
    temperature=0,                  # 0 = respuestas deterministas (ideal para análisis)
    api_key=MIAPIKEI
)

print(f"✅ Modelo cargado: {llm.model}")

# ── Demo rápida de .invoke() ───────────────────────────────────────────
print("\n🧪 Probando .invoke() directamente sobre el LLM...")
respuesta = llm.invoke("Di 'Hola, estoy listo para analizar datos' en español.")
print(f"\n🤖 Mistral dice: {respuesta.content}")

✅ Modelo cargado: mistral-large-latest

🧪 Probando .invoke() directamente sobre el LLM...

🤖 Mistral dice: La frase ya está en español:

**"Hola, estoy listo para analizar datos"**.

Si quieres ajustar el tono o el contexto, aquí tienes algunas variantes:

- **Formal:** *"Hola, estoy preparado/a para analizar los datos"*.
- **Coloquial:** *"¡Hola! Ya estoy listo/a para ponerme con los datos"*.
- **Técnico:** *"Hola, estoy listo para iniciar el análisis de datos"*.

¿Necesitas ayuda con algo más relacionado con el análisis de datos? 😊


## 📂 PASO 4 — Cargar tu Dataset

Tienes **3 opciones**:  
- **Opción A**: Subir un archivo desde tu PC (CSV, Excel, JSON)  
- **Opción B**: Cargar un dataset de ejemplo desde la web  
- **Opción C**: Pegar datos directamente en el código

In [1]:
import pandas as pd
import io

# ══════════════════════════════════════════════════════════════════════
# OPCIÓN A — Subir archivo desde tu computadora (recomendado en Colab)
# ══════════════════════════════════════════════════════════════════════
USAR_OPCION = "B"   # Cambia a "B" para usar dataset de ejemplo

if USAR_OPCION == "A":
    from google.colab import files
    print("📂 Selecciona tu archivo CSV, Excel o JSON:")
    uploaded = files.upload()

    filename = list(uploaded.keys())[0]
    content  = uploaded[filename]

    if filename.endswith(".csv"):
        df = pd.read_csv(io.BytesIO(content))
    elif filename.endswith((".xlsx", ".xls")):
        df = pd.read_excel(io.BytesIO(content))
    elif filename.endswith(".json"):
        df = pd.read_json(io.BytesIO(content))
    else:
        raise ValueError(f"Formato no soportado: {filename}")

    print(f"\n✅ Dataset '{filename}' cargado exitosamente")

# ══════════════════════════════════════════════════════════════════════
# OPCIÓN B — Dataset de ejemplo (Titanic)
# ══════════════════════════════════════════════════════════════════════
elif USAR_OPCION == "B":
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df  = pd.read_csv(url)
    print("✅ Dataset de ejemplo (Titanic) cargado")

# ── Vista previa ──────────────────────────────────────────────────────
print(f"\n📊 Forma del dataset: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"📋 Columnas: {list(df.columns)}")
print()
df.head()

✅ Dataset de ejemplo (Titanic) cargado

📊 Forma del dataset: 891 filas × 12 columnas
📋 Columnas: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']



,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 🔍 PASO 5 — Exploración rápida del Dataset

In [ ]:
print("="*60)
print("📊 RESUMEN ESTADÍSTICO")
print("="*60)
display(df.describe(include='all').T)

print("\n" + "="*60)
print("🔎 TIPOS DE DATOS Y VALORES NULOS")
print("="*60)
info_df = pd.DataFrame({
    "Dtype":       df.dtypes,
    "Nulos":       df.isnull().sum(),
    "% Nulos":     (df.isnull().mean() * 100).round(2),
    "Únicos":      df.nunique()
})
display(info_df)

## 🤖 PASO 6 — Crear el Agente Pandas

### ¿Qué es `create_pandas_dataframe_agent`?

Es una función de LangChain que crea un **agente ReAct** capaz de:
1. Leer tu DataFrame
2. Generar código Python/Pandas para responder preguntas
3. Ejecutar ese código y devolver el resultado

**Flujo interno cuando usas `.invoke()`:**
```
Tu pregunta
    ↓
[LLM] Razona y genera código Pandas
    ↓
[Tool: Python REPL] Ejecuta el código
    ↓
[LLM] Interpreta el resultado
    ↓
Respuesta en lenguaje natural
```

In [12]:
from langchain_experimental.agents import create_pandas_dataframe_agent

# ── Crear el agente ────────────────────────────────────────────────────
agent = create_pandas_dataframe_agent(
    llm                  = llm,
    df                   = df,
    agent_type           = "tool-calling",   # ✅ Compatible con Mistral
    verbose              = True,
    allow_dangerous_code = True
)

print("✅ Agente Pandas creado correctamente")
print(f"🛠️  Herramientas: {[t.name for t in agent.tools]}")

✅ Agente Pandas creado correctamente
🛠️  Herramientas: ['python_repl_ast']


## 💬 PASO 7 — Función de consulta con `.invoke()` explicado

In [13]:
def preguntar(pregunta: str, verbose: bool = True) -> str:
    """
    Función que usa .invoke() para hacerle una pregunta al agente.

    ¿Cómo funciona .invoke() aquí?
    ─────────────────────────────────────────────────────────────
    agent.invoke({"input": pregunta})
        │
        ├─ El agente recibe el dict {"input": pregunta}
        │
        ├─ Internamente LangChain ejecuta el loop ReAct:
        │     Thought → Action (código Pandas) → Observation → ...
        │
        └─ Devuelve un dict con:
              {"input": ..., "output": "respuesta final"}
    ─────────────────────────────────────────────────────────────
    """
    print(f"\n{'═'*60}")
    print(f"❓ Pregunta: {pregunta}")
    print(f"{'═'*60}\n")

    resultado = agent.invoke({"input": pregunta})   # ← .invoke() aquí

    respuesta = resultado["output"]

    print(f"\n{'─'*60}")
    print(f"🤖 Respuesta del agente:\n")
    print(f"   {respuesta}")
    print(f"{'─'*60}\n")

    return respuesta

print("✅ Función 'preguntar()' lista")

✅ Función 'preguntar()' lista


## 🧪 PASO 8 — Consultas de ejemplo al Agente

In [14]:
# ── Pregunta 1: Descripción general ───────────────────────────────────
preguntar("¿Cuántas filas y columnas tiene el dataset? Describe brevemente cada columna.")


════════════════════════════════════════════════════════════
❓ Pregunta: ¿Cuántas filas y columnas tiene el dataset? Describe brevemente cada columna.
════════════════════════════════════════════════════════════



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': 'df.shape'}`


(891, 12)El dataset `df` tiene **891 filas** (registros) y **12 columnas** (variables). A continuación, te describo brevemente cada columna:

1. **PassengerId**:
   - Identificador único de cada pasajero. Es un número entero.

2. **Survived**:
   - Indica si el pasajero sobrevivió o no al naufragio.
   - `0`: No sobrevivió.
   - `1`: Sobrevivió.

3. **Pclass**:
   - Clase del pasajero en el barco (representa el nivel socioeconómico).
   - `1`: Primera clase (alta).
   - `2`: Segunda clase (media).
   - `3`: Tercera clase (baja).

4. **Name**:
   - Nombre completo del pasajero, incluyendo títulos como Mr., Mrs., Miss, etc.

5. **Sex**:
   - Género del pasajero. Puede ser `male` 

'El dataset `df` tiene **891 filas** (registros) y **12 columnas** (variables). A continuación, te describo brevemente cada columna:\n\n1. **PassengerId**:\n   - Identificador único de cada pasajero. Es un número entero.\n\n2. **Survived**:\n   - Indica si el pasajero sobrevivió o no al naufragio.\n   - `0`: No sobrevivió.\n   - `1`: Sobrevivió.\n\n3. **Pclass**:\n   - Clase del pasajero en el barco (representa el nivel socioeconómico).\n   - `1`: Primera clase (alta).\n   - `2`: Segunda clase (media).\n   - `3`: Tercera clase (baja).\n\n4. **Name**:\n   - Nombre completo del pasajero, incluyendo títulos como Mr., Mrs., Miss, etc.\n\n5. **Sex**:\n   - Género del pasajero. Puede ser `male` (hombre) o `female` (mujer).\n\n6. **Age**:\n   - Edad del pasajero en años. Puede contener valores nulos (`nan`).\n\n7. **SibSp**:\n   - Número de hermanos o cónyuges que el pasajero tenía a bordo.\n\n8. **Parch**:\n   - Número de padres o hijos que el pasajero tenía a bordo.\n\n9. **Ticket**:\n   - 

In [15]:
# ── Pregunta 2: Estadísticas ───────────────────────────────────────────
preguntar("¿Cuál es el promedio, máximo y mínimo de las columnas numéricas?")


════════════════════════════════════════════════════════════
❓ Pregunta: ¿Cuál es el promedio, máximo y mínimo de las columnas numéricas?
════════════════════════════════════════════════════════════



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "# Calculating the mean, max, and min of the numeric columns in the dataframe\ndf.describe(include=[float, int]).loc[['mean', 'max', 'min']]"}`


      PassengerId  Survived    Pclass        Age     SibSp     Parch  \
mean        446.0  0.383838  2.308642  29.699118  0.523008  0.381594   
max         891.0  1.000000  3.000000  80.000000  8.000000  6.000000   
min           1.0  0.000000  1.000000   0.420000  0.000000  0.000000   

            Fare  
mean   32.204208  
max   512.329200  
min     0.000000  Aquí están los valores promedio, máximo y mínimo de las columnas numéricas del dataframe `df`:

|       | PassengerId | Survived | Pclass | Age       | SibSp | Parch | Fare     |
|-------|-------------|--

'Aquí están los valores promedio, máximo y mínimo de las columnas numéricas del dataframe `df`:\n\n|       | PassengerId | Survived | Pclass | Age       | SibSp | Parch | Fare     |\n|-------|-------------|----------|--------|-----------|-------|-------|----------|\n| **mean**  | 446.0       | 0.3838   | 2.31   | 29.699    | 0.523 | 0.382 | 32.204   |\n| **max**   | 891.0       | 1.0      | 3.0    | 80.0      | 8.0   | 6.0   | 512.329  |\n| **min**   | 1.0         | 0.0      | 1.0    | 0.42      | 0.0   | 0.0   | 0.0      |'

In [17]:
# ── Pregunta 3: Valores nulos ──────────────────────────────────────────
preguntar("¿Existen valores nulos o faltantes? ¿En qué columnas y cuántos?")


════════════════════════════════════════════════════════════
❓ Pregunta: ¿Existen valores nulos o faltantes? ¿En qué columnas y cuántos?
════════════════════════════════════════════════════════════



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': 'df.isnull().sum()'}`


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64En el dataframe `df`, las columnas con valores nulos o faltantes son las siguientes:

- **Age**: 177 valores faltantes.
- **Cabin**: 687 valores faltantes.
- **Embarked**: 2 valores faltantes.

> Finished chain.

────────────────────────────────────────────────────────────
🤖 Respuesta del agente:

   En el dataframe `df`, las columnas con valores nulos o faltantes son las siguientes:

- **Age**: 177 valores faltantes.
- **Cabin**: 687 valores falt

'En el dataframe `df`, las columnas con valores nulos o faltantes son las siguientes:\n\n- **Age**: 177 valores faltantes.\n- **Cabin**: 687 valores faltantes.\n- **Embarked**: 2 valores faltantes.'

In [18]:
# ── Pregunta 4: Correlaciones ──────────────────────────────────────────
preguntar("¿Qué columnas numéricas tienen mayor correlación entre sí?")


════════════════════════════════════════════════════════════
❓ Pregunta: ¿Qué columnas numéricas tienen mayor correlación entre sí?
════════════════════════════════════════════════════════════



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': '# Seleccionamos solo las columnas numéricas del dataframe\ndf_numeric = df.select_dtypes(include=[\'int64\', \'float64\'])\n\n# Calculamos la matriz de correlación\ncorrelation_matrix = df_numeric.corr()\n\n# Obtenemos las parejas de columnas con mayor correlación (excluyendo la diagonal)\ncorrelation_unstacked = correlation_matrix.where(~np.eye(len(correlation_matrix), dtype=bool)).unstack()\n\n# Ordenamos las correlaciones de mayor a menor\nsorted_correlation = correlation_unstacked.sort_values(kind="quicksort", ascending=False)\n\nsorted_correlation.head(10)  # Mostramos las 10 parejas con mayor correlación'}`


NameError: name 'np' is not defined

HTTPStatusError: Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}

In [20]:
# ── Pregunta 5: Top valores ────────────────────────────────────────────
preguntar("¿Cuáles son los 5 valores más frecuentes en cada columna categórica?")


════════════════════════════════════════════════════════════
❓ Pregunta: ¿Cuáles son los 5 valores más frecuentes en cada columna categórica?
════════════════════════════════════════════════════════════



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "import pandas as pd\n\n# Columnas categóricas en el dataframe\ncategorical_columns = ['Pclass', 'Sex', 'SibSp', 'Parch', 'Embarked', 'Cabin']\n\n# Obtener los 5 valores más frecuentes para cada columna categórica\nmost_frequent_values = {}\nfor column in categorical_columns:\n    if column in df.columns:\n        most_frequent_values[column] = df[column].value_counts().head(5).to_dict()\n\nmost_frequent_values"}`


{'Pclass': {3: 491, 1: 216, 2: 184}, 'Sex': {'male': 577, 'female': 314}, 'SibSp': {0: 608, 1: 209, 2: 28, 4: 18, 3: 16}, 'Parch': {0: 678, 1: 118, 2: 80, 5: 5, 3: 5}, 'Embarked': {'S': 644, 'C': 168, 'Q': 77}, 'Cabin': {'G6': 4, 'C23 C25 C27': 4, 'B96 B98': 4, 'F33': 3, 'E101': 3}}Aquí ti

'Aquí tienes los **5 valores más frecuentes** en cada columna categórica del dataframe `df`:\n\n1. **Pclass** (Clase del pasajero):\n   - 3: 491 veces\n   - 1: 216 veces\n   - 2: 184 veces\n\n2. **Sex** (Género):\n   - male (hombre): 577 veces\n   - female (mujer): 314 veces\n\n3. **SibSp** (Número de hermanos/cónyuges a bordo):\n   - 0: 608 veces\n   - 1: 209 veces\n   - 2: 28 veces\n   - 4: 18 veces\n   - 3: 16 veces\n\n4. **Parch** (Número de padres/hijos a bordo):\n   - 0: 678 veces\n   - 1: 118 veces\n   - 2: 80 veces\n   - 5: 5 veces\n   - 3: 5 veces\n\n5. **Embarked** (Puerto de embarque):\n   - S: 644 veces\n   - C: 168 veces\n   - Q: 77 veces\n\n6. **Cabin** (Cabina):\n   - G6: 4 veces\n   - C23 C25 C27: 4 veces\n   - B96 B98: 4 veces\n   - F33: 3 veces\n   - E101: 3 veces\n\n---\n**Nota**:\n- La columna **Cabin** tiene muchos valores faltantes (`nan`), por lo que solo se muestran los valores más frecuentes que están presentes.\n- En columnas como **Sex** o **Embarked**, hay m

## 💬 PASO 9 — Chat interactivo con tus datos

Escribe tus propias preguntas en lenguaje natural:

In [23]:
# ── Escribe aquí tu pregunta personalizada ─────────────────────────────
MI_PREGUNTA = "¿Puedes hacer un resumen ejecutivo del dataset en 5 puntos clave?"

preguntar(MI_PREGUNTA)


════════════════════════════════════════════════════════════
❓ Pregunta: ¿Puedes hacer un resumen ejecutivo del dataset en 5 puntos clave?
════════════════════════════════════════════════════════════



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "# Verificar las dimensiones del dataset (filas, columnas)\nshape = df.shape\n\n# Verificar tipos de datos y valores nulos\ninfo = df.info()\n\n# Estadísticas descriptivas básicas de las columnas numéricas\nstats = df.describe(include='number')\n\n# Contar valores únicos en columnas categóricas relevantes\ncategorical_summary = {\n    'Survived': df['Survived'].value_counts(),\n    'Pclass': df['Pclass'].value_counts(),\n    'Sex': df['Sex'].value_counts(),\n    'Embarked': df['Embarked'].value_counts(),\n    'Cabin': df['Cabin'].notna().sum()  # Contar no nulos en Cabin\n}\n\n# Calcular la tasa de supervivencia general\nsurvival_rate = df['Survived'].mean() * 100\n\nshape, stats, categorical_summary, su

'Aquí tienes un **resumen ejecutivo** del dataset en **5 puntos clave**:\n\n---\n\n### 1. **Tamaño del dataset**\n   - Contiene **891 registros** (pasajeros) y **12 columnas** (variables como edad, género, clase, etc.).\n   - Representa aproximadamente el **40% del total de pasajeros del Titanic** (2,224 en la vida real).\n\n---\n\n### 2. **Supervivencia**\n   - **Tasa de supervivencia general**: **38.4%** (342 de 891 pasajeros sobrevivieron).\n   - La mayoría de los pasajeros **no sobrevivieron** (549 fallecidos).\n\n---\n\n### 3. **Perfil demográfico**\n   - **Género**:\n     - **65% hombres** (577) y **35% mujeres** (314).\n   - **Edad**:\n     - Promedio: **29.7 años** (rango de 0.4 a 80 años).\n     - El **25% de los pasajeros tenía menos de 20 años** y el **50% menos de 28 años**.\n   - **Clase (Pclass)**:\n     - **Clase 3 (económica)**: 55% de los pasajeros (491).\n     - **Clase 1 (alta)**: 24% (216).\n     - **Clase 2 (media)**: 21% (184).\n\n---\n\n### 4. **Datos faltantes y

In [ ]:
# ── Modo chat continuo ─────────────────────────────────────────────────
print("💬 Modo chat activado. Escribe 'salir' para terminar.\n")

while True:
    pregunta = input("Tú: ")
    if pregunta.lower() in ["salir", "exit", "quit"]:
        print("👋 ¡Hasta luego!")
        break
    if pregunta.strip():
        preguntar(pregunta)

💬 Modo chat activado. Escribe 'salir' para terminar.

Tú: hola

════════════════════════════════════════════════════════════
❓ Pregunta: hola
════════════════════════════════════════════════════════════



> Entering new AgentExecutor chain...
¡Hola! ¿En qué puedo ayudarte con tu dataframe `df` de pandas? 😊

Por ejemplo, puedo ayudarte con:
- Filtrar datos.
- Calcular estadísticas.
- Limpiar datos (valores nulos, duplicados, etc.).
- Crear nuevas columnas.
- Visualizar datos.
- O cualquier otra operación.

> Finished chain.

────────────────────────────────────────────────────────────
🤖 Respuesta del agente:

   ¡Hola! ¿En qué puedo ayudarte con tu dataframe `df` de pandas? 😊

Por ejemplo, puedo ayudarte con:
- Filtrar datos.
- Calcular estadísticas.
- Limpiar datos (valores nulos, duplicados, etc.).
- Crear nuevas columnas.
- Visualizar datos.
- O cualquier otra operación.
────────────────────────────────────────────────────────────



## 📚 PASO 10 — Bonus: Entendiendo `.invoke()` vs otras formas de llamar

LangChain ofrece 3 formas de ejecutar un componente Runnable:

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Construir una cadena simple para demostración ──────────────────────
prompt = ChatPromptTemplate.from_template(
    "Eres un analista de datos. Responde en español.\n\nPregunta: {pregunta}"
)
cadena = prompt | llm | StrOutputParser()

print("🔷 FORMA 1: .invoke()  → Ejecuta sincrónicamente, devuelve resultado completo")
resultado = cadena.invoke({"pregunta": "¿Qué es un DataFrame?"})
print(f"   Resultado: {resultado[:120]}...\n")

print("🔷 FORMA 2: .stream()  → Devuelve chunks en tiempo real (streaming)")
print("   Output: ", end="", flush=True)
for chunk in cadena.stream({"pregunta": "Di 'Streaming activado' solamente."}):
    print(chunk, end="", flush=True)
print("\n")

print("🔷 FORMA 3: .batch()   → Ejecuta múltiples inputs en paralelo")
inputs   = [
    {"pregunta": "¿Qué es la media?"},
    {"pregunta": "¿Qué es la mediana?"}
]
resultados = cadena.batch(inputs)
for i, r in enumerate(resultados):
    print(f"   Batch {i+1}: {r[:80]}...")

print("\n✅ Las tres formas usan la misma interfaz Runnable de LangChain")

---

## 📌 Resumen

| Concepto | Descripción |
|---|---|
| `ChatMistralAI` | Wrapper de LangChain para el modelo Mistral |
| `create_pandas_dataframe_agent` | Crea un agente capaz de razonar sobre tu DataFrame |
| `.invoke({"input": ...})` | Método estándar de LangChain para ejecutar agentes/cadenas |
| `AgentType.OPENAI_FUNCTIONS` | Tipo de agente que usa function calling (compatible con Mistral) |
| `verbose=True` | Muestra el razonamiento paso a paso del agente |

### 🔗 Recursos útiles
- [LangChain Docs](https://python.langchain.com)
- [Mistral AI Console](https://console.mistral.ai)
- [LangChain Pandas Agent](https://python.langchain.com/docs/integrations/toolkits/pandas/)